# Macro Metrics vs Train-Size Ratio

This notebook scans `evaluation_metrics.txt` files from runs like:

- `out/seed*/train_use_*/BySeries_global/evaluation_metrics.txt`
- `out/seed*/train_use_*/BySeries_locals/evaluation_metrics.txt`

It then averages micro and macro metrics over seeds at each train-size ratio and plots separate charts per metric for each average type.

In [ ]:
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Figure widths matching typical two-column paper layouts
SINGLE_COL = (3.5, 2.5)  # ~88mm — fits one column (default)
DOUBLE_COL = (6.5, 3.5)  # ~165mm — spans both columns

mpl.rcParams.update(
    {
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "pgf.preamble": r"\usepackage{amsfonts}\usepackage{amssymb}\usepackage{amsmath}",
        "lines.linewidth": 1,
        "figure.figsize": SINGLE_COL,
        "font.size": 9,
        "savefig.dpi": 300,
    }
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

In [ ]:
def find_out_dir(start: Path | None = None) -> Path:
    """Find the nearest `out/` directory from current working directory upward."""
    start = (start or Path.cwd()).resolve()
    for candidate_root in [start, *start.parents]:
        out_dir = candidate_root / "out"
        if out_dir.exists() and out_dir.is_dir():
            return out_dir
    raise FileNotFoundError("Could not find an `out/` directory from current path upward.")


OUT_DIR = find_out_dir()
METRICS_GLOBS = {
    "Global": "seed*/train_use_*/Forecasting_global/evaluation_metrics.txt",
    "Locals": "seed*/train_use_*/Forecasting_locals/evaluation_metrics.txt",
    "Chronos": "seed*/train_use_*/Forecasting_CHRONOS/evaluation_metrics.txt",
}
SHOW_STD_BAND = True  # set False to hide +/- 1 std shading

metric_files = []
for model_scope, pattern in METRICS_GLOBS.items():
    for path in sorted(OUT_DIR.glob(pattern)):
        metric_files.append((model_scope, path))

print(f"Using out dir: {OUT_DIR}")
print(f"Found {len(metric_files)} matching metrics files")
metric_files[:5]

In [ ]:
SEED_RE = re.compile(r"^seed(\d+)$")
TRAIN_RE = re.compile(r"^train_use_(\d+)$")


def parse_seed_and_ratio(path: Path) -> tuple[int, float, int]:
    seed = None
    ratio_pct = None

    for part in path.parts:
        m_seed = SEED_RE.match(part)
        if m_seed:
            seed = int(m_seed.group(1))

        m_ratio = TRAIN_RE.match(part)
        if m_ratio:
            ratio_pct = int(m_ratio.group(1))

    if seed is None or ratio_pct is None:
        raise ValueError(f"Could not parse seed/ratio from: {path}")

    return seed, ratio_pct / 100.0, ratio_pct


def rows_from_metrics_file(path: Path, model_scope: str) -> list[dict]:
    seed, train_ratio, train_ratio_pct = parse_seed_and_ratio(path)
    df = pd.read_csv(path)

    metric_cols = [c for c in df.columns if c != "Series_ID"]
    macro_row = df.loc[df["Series_ID"] == "Macro_Average"]
    micro_row = df.loc[df["Series_ID"] == "Micro_Average"]

    if macro_row.empty or micro_row.empty:
        raise ValueError(f"Missing Macro_Average/Micro_Average rows in: {path}")

    out_rows = []
    for metric_name in metric_cols:
        out_rows.append(
            {
                "seed": seed,
                "train_ratio": train_ratio,
                "train_ratio_pct": train_ratio_pct,
                "metric": metric_name,
                "average_type": "Macro",
                "model_scope": model_scope,
                "value": float(macro_row.iloc[0][metric_name]),
                "path": str(path),
            }
        )
        out_rows.append(
            {
                "seed": seed,
                "train_ratio": train_ratio,
                "train_ratio_pct": train_ratio_pct,
                "metric": metric_name,
                "average_type": "Micro",
                "model_scope": model_scope,
                "value": float(micro_row.iloc[0][metric_name]),
                "path": str(path),
            }
        )

    return out_rows


all_rows = []
for model_scope, file_path in metric_files:
    all_rows.extend(rows_from_metrics_file(file_path, model_scope))

metrics_long = pd.DataFrame(all_rows)
metrics_long = metrics_long.sort_values(["metric", "average_type", "model_scope", "train_ratio", "seed"]).reset_index(drop=True)

print(f"Loaded {len(metrics_long)} rows")
metrics_long.head(12)

In [ ]:
seed_coverage = (
    metrics_long.groupby(["train_ratio", "average_type"])["seed"]
    .nunique()
    .rename("n_seeds")
    .reset_index()
    .sort_values(["train_ratio", "average_type"])
)
seed_coverage

In [ ]:
agg = (
    metrics_long.groupby(["metric", "average_type", "model_scope", "train_ratio", "train_ratio_pct"], as_index=False)
    .agg(
        mean_value=("value", "mean"),
        std_value=("value", "std"),
        n_seeds=("seed", "nunique"),
    )
    .sort_values(["metric", "average_type", "model_scope", "train_ratio"])
)

agg.head(20)

In [ ]:
metrics = sorted(agg["metric"].unique())
better_direction = {
    "LogLik": "up",
    "RMSE": "down",
    "MAE": "down",
    "P50": "down",
    "P90": "down",
}
line_styles = {
    "Global": {"color": "tab:blue", "label": "Global"},
    "Locals": {"color": "tab:red", "label": "Local"},
    "Chronos": {"color": "tab:green", "label": "Chronos-2"},
}
average_types = ["Macro", "Micro"]
x_ticks = sorted(agg["train_ratio"].unique())

for metric in metrics:
    for average_type in average_types:
        fig, ax = plt.subplots(figsize=DOUBLE_COL)

        for model_scope, style in line_styles.items():
            sub = (
                agg[
                    (agg["metric"] == metric)
                    & (agg["average_type"] == average_type)
                    & (agg["model_scope"] == model_scope)
                ]
                .sort_values("train_ratio")
                .reset_index(drop=True)
            )
            if sub.empty:
                continue

            x = sub["train_ratio"].to_numpy()
            y = sub["mean_value"].to_numpy()
            y_std = sub["std_value"].fillna(0.0).to_numpy()

            ax.plot(x, y, marker="o", color=style["color"], label=style["label"])
            if SHOW_STD_BAND:
                ax.fill_between(x, y - y_std, y + y_std, color=style["color"], alpha=0.15)

        ax.set_xlabel("Train-size ratio")
        ax.set_ylabel(metric)
        ax.set_xticks(x_ticks)
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15),
                  ncol=3, frameon=False)
        fig.tight_layout()

        fname = f"{average_type}_{metric}_vs_train_ratio"
        fig.savefig(OUT_DIR / f"{fname}.pdf", bbox_inches="tight")
        fig.savefig(OUT_DIR / f"{fname}.svg", bbox_inches="tight")
        plt.show()

In [ ]:
# # Optional: save aggregated table for later use
# save_path = OUT_DIR / "macro_micro_metrics_avg_over_seeds.csv"
# agg.to_csv(save_path, index=False)
# print(f"Saved: {save_path}")